<div align = "center">

# ***Crop Productivity Data Cleaning***

</div>

## Dataset description

| Column | Meaning |
|---|---|
| `Crop` | Crop name |
| `GOV` | Egyptian governorate |
| `Year` | Cultivation year |
| `Session` | Agricultural season (`Winter`, `Summer`, `Perennial`) |
| `Area` | Cultivated area |
| `Production` | Total production |
| `Yield` | Production per unit of area (equal `Production / Area`) |


## ***Importing Libraries***

In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

## ***Data Loading & Overview:***

### 1. **Loading the Dataset**

In [28]:
def load_data(file_path):
    """ load the dataset from excel file and return dataframe """
    try:
        data = pd.read_excel(file_path)
    except Exception as e:
        print(f"file not found error: {e}")
        return None
    else:
        print("data loaded successfully")
        return data
    finally:
        print("Execution completed.")

file_path = "../Data/Raw Data/Egypt Crops.xlsx"
df = load_data(file_path)
df.head()

data loaded successfully
Execution completed.


,Crop,GOV,Year,Season,Area,Production,Yield
0,Wheat,Alexandria,2010,Winter,60424.0,148099.0,2.45
1,Wheat,Beheira,2010,Winter,293263.0,767616.0,2.62
2,Wheat,Gharbiya,2010,Winter,147883.0,390633.0,2.64
3,Wheat,Kafr El Sheikh,2010,Winter,234816.0,598429.0,2.55
4,Wheat,Dakahlia,2010,Winter,296392.0,754910.0,2.55


### 2. ***Data Overview***

In [29]:
print("Dataset Shape:", df.shape)

Dataset Shape: (18998, 7)


In [30]:
df.columns.tolist()

['Crop', 'GOV', 'Year', 'Season', 'Area', 'Production', 'Yield']

In [31]:
print("\nDataset Info:")
df.info()


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18998 entries, 0 to 18997
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Crop        18998 non-null  object 
 1   GOV         18998 non-null  object 
 2   Year        18998 non-null  int64  
 3   Season      18998 non-null  object 
 4   Area        18998 non-null  float64
 5   Production  18998 non-null  float64
 6   Yield       18998 non-null  float64
dtypes: float64(3), int64(1), object(3)
memory usage: 1.0+ MB


In [32]:
numeric_cols = ['Area', 'Production', 'Yield']

print("Statistical Analysis:")
df[numeric_cols].describe().T

Statistical Analysis:


,count,mean,std,min,25%,50%,75%,max
Area,18998.0,10102.749914,34408.949142,0.0,0.0,262.000000,3086.000000,436506.0
Production,18998.0,84574.588300,390021.823724,0.0,0.0,1067.000000,19227.750000,8600009.0
Yield,18998.0,12.416221,191.038592,0.0,0.0,3.345124,8.319825,8035.0


In [33]:
print("\nStatistical Summary for Categorical Variables:")
df.describe(include=object).T


Statistical Summary for Categorical Variables:


,count,unique,top,freq
Crop,18998,58,Sesame,420
GOV,18998,29,Alexandria,681
Season,18998,3,Summer,7026


## ***Data Quality Check & Cleaning***

### 1. ***Data Quality Check***

#### **Missing Values**

In [34]:
print("Missing Values:")
df.isnull().sum()

Missing Values:


Crop          0
GOV           0
Year          0
Season        0
Area          0
Production    0
Yield         0
dtype: int64

#### **Duplicated Rows**

In [35]:
print("Full duplicates:", df.duplicated().sum())
print("Duplicate business keys:", 
    df.duplicated(
        subset=['Crop', 'GOV', 'Year', 'Season']).sum())

Full duplicates: 0
Duplicate business keys: 0


#### **Check empty fields**


In [36]:
print("\nEmpty or Whitespace Values:")
for col in df.select_dtypes(include='object').columns:
    empty_count = df[col].apply(
        lambda x: isinstance(x, str) and x.strip() == "").sum()
    print(f"{col}: {empty_count}")

missing_percentage = (df.isnull().sum() / len(df)) * 100



Empty or Whitespace Values:
Crop: 0
GOV: 0
Season: 0


In [37]:
categorical_columns = ['Crop', 'GOV', 'Season']
for col in categorical_columns:
    normalized = df[col].str.strip().str.lower()
    comparison = pd.DataFrame({
        'Original': df[col],
        'Normalized': normalized}).drop_duplicates()
    variants = comparison.groupby('Normalized')['Original'].nunique()
    print(f"\n{col} - Possible formatting issues:")
    print(variants[variants > 1])


Crop - Possible formatting issues:
Series([], Name: Original, dtype: int64)



GOV - Possible formatting issues:
Normalized
south sinai    2
Name: Original, dtype: int64

Season - Possible formatting issues:
Series([], Name: Original, dtype: int64)


In [38]:
df.GOV.unique()

array(['Alexandria', 'Beheira', 'Gharbiya', 'Kafr El Sheikh', 'Dakahlia',
       'Damietta', 'Sharqiya', 'Ismailia', 'Suez', 'Monufia', 'Qalyoubia',
       'Cairo', 'Giza', 'Bani Suef', 'Fayoum', 'Minya', 'Asyut', 'Sohag',
       'Qena', 'Luxor', 'Aswan', 'New Vally', 'Marsa Matrouh',
       'North Sinai', 'Port Said', 'Red Sea', 'South sinai',
       'South Sinai', 'Noubariya'], dtype=object)

In [39]:
df.Crop.unique()

array(['Wheat', 'Barley', 'White Maize', 'Sorghum', 'Yellow Maize',
       'Rice', 'Sugar Beet', 'Sugar Cane', 'Peanuts', 'Sesame', 'Soybean',
       'Sunflower', 'Flax', 'Cotton', 'Alfalfa', 'Beans', 'Cabbage',
       'Cantaloupe', 'Cauliflower', 'Chickpeas', 'Clover', 'Cucumber',
       'Eggplant', 'Fenugreek', 'Garlic', 'Green Beans', 'Green Fodder',
       'Green Peas', 'Jews Mallow', 'Lentil', 'Lupine',
       'Medical, Aromatic Plant', 'Minor Vegetables', 'Okra', 'Onion',
       'Pepper', 'Potatoes', 'Roots vegetables', 'Squash', 'Strawberry',
       'Sweet Potatoes', 'Tomato', 'Water Melon', 'Apple', 'Apricot',
       'Grapes', 'Guava', 'Lemon', 'Mandarin', 'Olive', 'Oranges',
       'Palm Trees', 'Pear', 'Pomegranate', 'Mango', 'Bananas', 'Figs',
       'Peaches'], dtype=object)

In [40]:
print("\nZero Values:")
print((df[numeric_cols] == 0).sum())


Zero Values:
Area          4795
Production    4852
Yield         4885
dtype: int64


In [41]:
print("\nNegative Values:")
print((df[numeric_cols] < 0).sum())


Negative Values:
Area          0
Production    0
Yield         0
dtype: int64


In [42]:
area_without_production = df[
    (df['Area'] > 0) &
    (df['Production'] == 0)].copy()
area_without_production[
        ['Crop', 'GOV', 'Year', 'Season', 'Area', 'Production', 'Yield']].sort_values(['Crop', 'Year', 'GOV'])

print("Total records:", len(area_without_production))
area_without_production.head(10)

Total records: 89


,Crop,GOV,Year,Season,Area,Production,Yield
350,Wheat,North Sinai,2022,Winter,20.0,0.0,0.0
378,Wheat,North Sinai,2023,Winter,102.0,0.0,0.0
3456,Peanuts,South sinai,2015,Summer,2.0,0.0,0.0
6496,Cabbage,Marsa Matrouh,2021,Winter,15.0,0.0,0.0
8861,Green Fodder,Noubariya,2015,Summer,25412.0,0.0,0.0
8889,Green Fodder,Noubariya,2016,Summer,41370.0,0.0,0.0
8902,Green Fodder,Monufia,2017,Summer,92560.0,0.0,0.0
8917,Green Fodder,Noubariya,2017,Summer,66150.0,0.0,0.0
8973,Green Fodder,Noubariya,2019,Summer,37631.0,0.0,0.0
8975,Green Fodder,Marsa Matrouh,2020,Summer,1.0,0.0,0.0


In [43]:
area_without_production['Crop'].value_counts().to_frame('Count')

,Count
Crop,
Figs,39
"Medical, Aromatic Plant",26
Green Fodder,7
Peaches,4
Apricot,3
Wheat,2
Lupine,1
Lentil,1
Cabbage,1


In [44]:
zero_production = df[(df['Area'] > 0) & (df['Production'] == 0)]

summary = zero_production.groupby(['Crop', 'GOV']).agg({
    'Year': lambda x: sorted(x.unique()),
    'Season': lambda x: sorted(x.unique()),
    'Area': 'sum',
    'Yield': 'mean'})
summary['Zero_Records'] = zero_production.groupby(['Crop', 'GOV']).size()

summary.sort_values('Zero_Records', ascending=False)

Year  \
Crop                    GOV                                                                 
Figs                    Damietta        [2010, 2011, 2012, 2013, 2014, 2015, 2016, 201...   
                        Dakahlia         [2010, 2011, 2012, 2013, 2014, 2015, 2019, 2020]   
Medical, Aromatic Plant South sinai            [2015, 2016, 2017, 2018, 2019, 2020, 2021]   
Figs                    Port Said                    [2011, 2012, 2013, 2014, 2015, 2016]   
Medical, Aromatic Plant Cairo                              [2015, 2016, 2017, 2018, 2019]   
                        Suez                               [2015, 2016, 2017, 2018, 2019]   
Green Fodder            Noubariya                                [2015, 2016, 2017, 2019]   
Figs                    Kafr El Sheikh                           [2011, 2012, 2013, 2014]   
Medical, Aromatic Plant Port Said                                      [2015, 2016, 2017]   
Figs                    Gharbiya                                       [2010, 2011, 2012]   
Apricot                 Dakahlia                                             [2022, 2023]   
Wheat                   North Sinai                                          [2022, 2023]   
Medical, Aromatic Plant Sohag                                                [2015, 2020]   
                        Alexandria                                           [2016, 2017]   
Peaches                 Damietta                                             [2021, 2022]   
Figs                    Monufia                                              [2015, 2016]   
Peaches                 Fayoum                                               [2022, 2023]   
Lentil                  New Vally                                                  [2019]   
Green Fodder            South sinai                                                [2020]   
                        Monufia                                                    [2017]   
                        Marsa Matrouh                                              [2020]   
Figs                    Luxor                                                      [2015]   
Cabbage                 Marsa Matrouh                                              [2021]   
Bananas                 Fayoum                                                     [2022]   
Apricot                 North Sinai                                                [2023]   
Figs                    Alexandria                                                 [2019]   
Medical, Aromatic Plant Red Sea                                                    [2022]   
Lupine                  Dakahlia                                                   [2015]   
Medical, Aromatic Plant Dakahlia                                                   [2015]   
Palm Trees              Red Sea                                                    [2020]   
Okra                    Marsa Matrouh                                              [2021]   
Peanuts                 South sinai                                                [2015]   
Pear                    Dakahlia                                                   [2021]   

                                             Season      Area  Yield  \
Crop                    GOV                                            
Figs                    Damietta        [Perennial]     371.0    0.0   
                        Dakahlia        [Perennial]   31080.0    0.0   
Medical, Aromatic Plant South sinai     [Perennial]     582.0    0.0   
Figs                    Port Said       [Perennial]      46.0    0.0   
Medical, Aromatic Plant Cairo           [Perennial]     495.0    0.0   
                        Suez            [Perennial]      38.0    0.0   
Green Fodder            Noubariya          [Summer]  170563.0    0.0   
Figs                    Kafr El Sheikh  [Perennial]   13916.0    0.0   
Medical, Aromatic Plant Port Said       [Perennial]      21.0    0.0   
Figs                    Gharbiya     

In [45]:
production_without_area = df[
    (df['Area'] == 0) &
    (df['Production'] > 0)].copy()
production_without_area[
        ['Crop', 'GOV', 'Year', 'Season', 'Area', 'Production', 'Yield']].sort_values(['Crop', 'Year', 'GOV'])

print("Total records:", len(production_without_area))
production_without_area.head(10)

Total records: 32


,Crop,GOV,Year,Season,Area,Production,Yield
6847,Cauliflower,Aswan,2016,Winter,0.0,21.0,0.0
9032,Green Fodder,North Sinai,2022,Summer,0.0,473.0,0.0
10792,Okra,South sinai,2021,Summer,0.0,36.0,0.0
10820,Okra,South sinai,2022,Summer,0.0,8.0,0.0
16274,Palm Trees,Port Said,2010,Perennial,0.0,560.0,0.0
16280,Palm Trees,South sinai,2010,Perennial,0.0,745.0,0.0
16302,Palm Trees,Port Said,2011,Perennial,0.0,542.0,0.0
16308,Palm Trees,South sinai,2011,Perennial,0.0,744.0,0.0
16330,Palm Trees,Port Said,2012,Perennial,0.0,634.0,0.0
16336,Palm Trees,South sinai,2012,Perennial,0.0,3281.0,0.0


In [46]:
production_without_area['Crop'].value_counts().to_frame('Count')

,Count
Crop,
Palm Trees,26
Figs,2
Okra,2
Cauliflower,1
Green Fodder,1


In [47]:
valid_records = df[(df['Area'] > 0) & (df['Production'] > 0)].copy()

valid_records['Calculated_Yield'] = (
    valid_records['Production'] / valid_records['Area'])

valid_records['Yield_Difference'] = (
    valid_records['Yield'] - valid_records['Calculated_Yield']).abs()


mismatch = valid_records[
    ~np.isclose(
        valid_records['Yield'],
        valid_records['Calculated_Yield'],
        rtol=1e-4,
        atol=0.01)]

print("Total valid records:", len(valid_records))
print("Matching records:", len(valid_records) - len(mismatch))
print("Mismatching records:", len(mismatch))

display(mismatch[
        ['Crop', 'GOV', 'Year', 'Season',
        'Area', 'Production', 'Yield',
        'Calculated_Yield', 'Yield_Difference']])

Total valid records: 14114
Matching records: 14113
Mismatching records: 1


,Crop,GOV,Year,Season,Area,Production,Yield,Calculated_Yield,Yield_Difference
382,Wheat,Cairo,2024,Winter,21.0,45.0,0.0,2.142857,2.142857


### 2. ***Data Cleaning***

In [48]:
df_cleaned = df.copy()

In [49]:
gov_mapping = {'South sinai': 'South Sinai'}

df_cleaned['GOV'] = df_cleaned['GOV'].replace(gov_mapping)
print("Unique Governorates After Cleaning:",df_cleaned['GOV'].nunique())
print("\nGovernorate Frequencies:")
df_cleaned['GOV'].value_counts().to_frame('Count')

Unique Governorates After Cleaning: 28

Governorate Frequencies:


,Count
GOV,
Alexandria,681
Beheira,681
Gharbiya,681
Kafr El Sheikh,681
Dakahlia,681
Damietta,681
Sharqiya,681
Ismailia,681
Suez,681


In [50]:
df_cleaned.loc[382, 'Yield'] = (
    df_cleaned.loc[382, 'Production'] /
    df_cleaned.loc[382, 'Area'])

display(
    df_cleaned.loc[[382], [
        'Crop', 'GOV', 'Year',
        'Area', 'Production',
        'Yield', 'Yield']])

,Crop,GOV,Year,Area,Production,Yield,Yield
382,Wheat,Cairo,2024,21.0,45.0,2.142857,2.142857


In [51]:
df_cleaned.columns = (
    df_cleaned.columns.str.strip()
    .str.replace(" ","_", regex=False)
    .str.replace("-","_", regex=False)
    .str.replace("/","_", regex=False)
    .str.lower())

print(df_cleaned.columns)

Index(['crop', 'gov', 'year', 'season', 'area', 'production', 'yield'], dtype='object')


In [52]:
df_cleaned.to_csv('../Data/Clean Data/Agricultural_Data_Cleaned.csv', index=False, encoding='utf-8-sig')

print("CSV file saved successfully!")
print("Shape:", df_cleaned.shape)

CSV file saved successfully!
Shape: (18998, 7)
